# שיעור 03 - תבניות עיצוב סוכניות  

בשיעור זה, נחקור שלוש תבניות עיצוב יסודיות לבניית סוכנים חכמים ויעילים:  

1. **הוראות ברורות לסוכן** — יצירת הנחיות מדויקות המגדירות את תפקיד הסוכן ומנחות את התנהגותו  
2. **פלט מובנה עם מודלי Pydantic** — הבטחת החזרת נתונים צפויים ומאומתים על ידי הסוכן  
3. **סוכנים בעלי אחריות יחידה** — תכנון סוכנים ממוקדים שכל אחד מבצע משימה אחת היטב  

ניישם כל תבנית על תרחיש של **ממליץ ליעדי נסיעה**, כאשר נבנה בהדרגה מערכת שיכולה להציע יעדים, לבדוק זמינות ולטפל בלוגיסטיקה.  


## התקנה


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic python-dotenv --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## תבנית 1: הנחיות ברורות לסוכן  

התבנית המשפיעה ביותר היא גם הפשוטה ביותר: כתיבת הנחיות ברורות ומפורטות עבור הסוכן שלך.  

הנחיות טובות מגדירות:  
- **מי** הסוכן (אישיות וטון)  
- **מה** עליו לעשות (אחריות שלב אחר שלב)  
- **איך** עליו להתנהג (הגבלות וסגנון)  

להלן, אנו יוצרים סוכן קונסיירז' נסיעות עם הנחיות מפורשות שמעצבות כל תגובה שהוא מייצר.  


In [ ]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## תבנית 2: פלט מובנה עם מודלים של Pydantic

טקסט חופשי שימושי לשיחה, אך מערכות בהמשך זקוקות לנתונים מובנים.
על ידי שילוב של **מודלים של Pydantic** עם **פונקציית כלי**, אנו יכולים:

- להגדיר סכימה מדויקת עבור הפלט של הסוכן
- לאמת תגובות באופן אוטומטי
- לשלב את תוצאות הסוכן בלוגיקת היישום בצורה אמינה

המפתח לאכיפה הוא העברת `response_format` כשמריצים את הסוכן. זה מאלץ את
המודל להחזיר אובייקט מאומת מסוג `TravelRecommendations` (זמין ב- `response.value`)
במקום טקסט חופשי. כלי `get_destination_details` גם מחזיר
`DestinationRecommendation` טיפוסי, כך שהנתונים נשארים מובנים מקצה לקצה.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(
    destination: Annotated[str, "The destination to look up"]
) -> DestinationRecommendation:
    """Get structured details about a vacation destination."""
    details = {
        "Barcelona": DestinationRecommendation(
            destination="Barcelona",
            available=True,
            best_season="May-Jun",
            highlights=["Beach", "Architecture", "Nightlife"],
            estimated_budget_usd=2000,
        ),
        "Tokyo": DestinationRecommendation(
            destination="Tokyo",
            available=True,
            best_season="Mar-Apr",
            highlights=["Culture", "Food", "Technology"],
            estimated_budget_usd=2500,
        ),
        "Cape Town": DestinationRecommendation(
            destination="Cape Town",
            available=False,
            best_season="Nov-Mar",
            highlights=["Nature", "Wine", "Adventure"],
            estimated_budget_usd=1800,
        ),
    }
    return details.get(
        destination,
        DestinationRecommendation(
            destination=destination,
            available=False,
            best_season="Unknown",
            highlights=[],
            estimated_budget_usd=0,
        ),
    )


structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

# Passing `response_format` forces the agent to return a validated
# TravelRecommendations object instead of free-form text.
response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget",
    options={"response_format": TravelRecommendations},
)

if response and response.value:
    result: TravelRecommendations = response.value
    for rec in result.recommendations:
        status = "Available" if rec.available else "Not available"
        print(f"{rec.destination} ({status})")
        print(f"  Best season: {rec.best_season}")
        print(f"  Highlights: {', '.join(rec.highlights)}")
        print(f"  Estimated budget: ${rec.estimated_budget_usd}")
        print()
    print(f"Note: {result.personalized_note}")
else:
    print("No validated structured response was returned.")
    print(response)


## תבנית 3: סוכנים עם אחריות יחידה

משימות מורכבות נהנות מפיצול העבודה בין סוכנים ממוקדים מרובים, כל אחד עם אחריות יחידה:

- **מומחה יעד** שיודע על מקומות וזמינות
- **מתכנן לוגיסטיקה** המטפל בטיסות, במלונות ובמסלולים

זה משקף את עקרון ההנדסת תוכנה של *הפרדת תחומים* — כל סוכן נגיש יותר לבדיקה, תחזוקה ושיפור בצורה עצמאית.


In [ ]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## סיכום

בשיעור זה יישמנו שלושה תבניות עיצוב מפעילות לתרחיש של ממליץ נסיעות:

| תבנית | רעיון מרכזי | יתרון |
|---|---|---|
| **הוראות ברורות** | הגדרת פרסונה, אחריות ומגבלות מראש | התנהגות סוכנים עקבית, בהתאם למותג |
| **פלט מובנה** | שימוש בדגמי Pydantic כפורמט התגובה | תוצאות מאומתות, קריאות למכונה |
| **אחריות יחידה** | כל סוכן מקבל משימה ממוקדת אחת | קל יותר לבדוק, לתחזק ולרכוב |

תבניות אלו מתחברות באופן טבעי — ניתן לשלב הוראות ברורות עם פלט מובנה בתוך סוכן עם אחריות יחידה, כדי לבנות מערכות חזקות ומוכנות לייצור.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**כתב ויתור**:
מסמך זה תורגם באמצעות שירות תרגום אוטומטי [Co-op Translator](https://github.com/Azure/co-op-translator). למרות שאנו שואפים לדיוק, יש לקחת בחשבון שתרגומים אוטומטיים עלולים להכיל שגיאות או אי-דיוקים. יש להחשיב את המסמך המקורי בשפתו הטבעית כמקור הסמכות. למידע קריטי מומלץ להשתמש בתרגום מקצועי על ידי מתרגם אדם. אנו לא אחראים לכל אי-הבנה או פירוש שגוי הנובע מהשימוש בתרגום זה.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
